# Compare Google Sheets Revision API Methods

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/investigate-oasis-sheets-aYOlG/notebooks/compare-api-methods.ipynb)

## Purpose

We have two API methods that claim to return historical Google Sheets revisions:

- **Method B:** Drive API v2 `revisions/{id}` → `exportLinks` → ODS download
- **Method C:** Direct Sheets URL: `spreadsheets/export?id=...&revision=N&exportFormat=ods`

A third method (Drive API v3 `files.export` with `revision` param) is KNOWN BROKEN
for Google Sheets — it silently ignores the revision parameter.

## Key Questions

1. Do Methods B and C return the same content for the same revision number?
2. Does Method C return genuinely historical content, or does it sometimes
   silently fall back to the current version?
3. What are the correct revision numbers for CI artifact versions V2, V3, V6
   (where the current mapping is wrong)?

## Background

Phase 1 verification found that 3 out of 10 revision-to-workflow mappings
are wrong:
- **V2:** Mapped to lib=1843 but CI V2 has different data (customs rewrite)
- **V3:** Mapped to lib=1868/doc=1803 but CI V3 has different doc data
- **V6:** Mapped to lib=1999 but CI V6 has different lib data

The Drive API `revisions.list` only returns ~25 "major" revisions, but there
are ~2005 internal revision save-points. The correct revisions are somewhere
between the major ones.

## Step 0: Authentication

In [ ]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request as AuthRequest

creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

## Step 1: Setup & Helpers

In [ ]:
import json, hashlib, time, zipfile, io, re
from urllib.request import Request, urlopen
from urllib.error import HTTPError

# --- Sheet IDs ---
SHEETS = {
    'ubl25_library':   '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    'ubl25_documents': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
}

ODS_MIME = 'application/x-vnd.oasis.opendocument.spreadsheet'
ODS_MIME_ALT = 'application/vnd.oasis.opendocument.spreadsheet'

# --- Known CI data hashes (Identification block stripped) ---
# These are the DEFINITIVE fingerprints from CI artifacts
CI_DATA_HASHES = {
    'V1':  'e4e961c1618ff112',  # Nov 17 10:42
    'V2':  'c21c9fd6de75cfe5',  # Nov 19 09:15 (customs rewrite)
    'V3':  '85d713818f2654c2',  # Nov 20 13:50 (WasteMovement added)
    'V4':  '94aa3b46e274e2dc',  # Nov 20 14:05 (NDR fix)
    'V5':  '93a197bc3b741bca',  # Dec 3 (CSD02 published)
    'V6':  '1ebde1fcadd0f0f9',  # Jan 21 16:38
    'V7':  '6504ba8eb96fb858',  # Jan 21 17:01
    'V8':  'cfaf508484d0f556',  # Jan 21 19:26
    'V9':  '5784456984fb5f12',  # Feb 9 14:42
    'V10': '5784456984fb5f12',  # Feb 9 14:46 (same data, different stage)
}

# Reverse lookup: data hash -> CI version(s)
CI_HASH_TO_VER = {}
for ver, h in CI_DATA_HASHES.items():
    CI_HASH_TO_VER.setdefault(h, []).append(ver)

# --- Reference content.xml hashes from Method B downloads ---
# These were extracted from the ODS files in work-sheets/revision-ods/
CONTENT_XML_REFS = {
    'ubl25_library': {
        '1843': 'ac8eb8226a3bdceed04fa9ce',
        '1868': '69ed22bf6746fd5a84ccf387',
        '1999': 'dec0bfa6a9a7f2574aa5d232',
        '2005': '716eacf3ebe60d7622e4c9c4',
    },
    'ubl25_documents': {
        '1793': 'f33306fcfe9227d89bbe8ecb',
        '1803': '7d43056bc165dfe5d95ea6ea',
        '1983': '8046f7e9d37de3d0d2e3f835',
        '2190': 'a24df7fa42e69416a8eb9de6',
        '2200': '90eea4287a813eac5fbafa6b',
        '2204': '952a3364f23d6fabeb3cdcac',
    },
}


def authenticated_get(url, binary=True):
    """Authenticated GET with retry."""
    headers = {'Authorization': f'Bearer {TOKEN}'}
    for attempt in range(4):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                return resp.status, resp.read()
        except HTTPError as e:
            if e.code in (429, 500, 502, 503):
                wait = 2 ** (attempt + 1)
                print(f'  retry({e.code})...', end='')
                time.sleep(wait)
                continue
            return e.code, None
        except Exception:
            if attempt < 3:
                time.sleep(2 ** (attempt + 1))
                continue
            return 0, None
    return 0, None


def method_b_download(file_id, rev_id):
    """Method B: Drive API v2 revisions/{id} -> exportLinks -> ODS."""
    # Step 1: Get revision metadata with exportLinks
    url = f'https://www.googleapis.com/drive/v2/files/{file_id}/revisions/{rev_id}'
    status, body = authenticated_get(url)
    if status != 200 or not body:
        return None, f'v2 metadata HTTP {status}'
    data = json.loads(body)
    export_links = data.get('exportLinks', {})
    ods_url = export_links.get(ODS_MIME) or export_links.get(ODS_MIME_ALT)
    if not ods_url:
        return None, f'No ODS exportLink (formats: {list(export_links.keys())})'

    # Step 2: Download ODS
    time.sleep(0.5)
    status, ods_data = authenticated_get(ods_url)
    if status != 200 or not ods_data:
        return None, f'ODS download HTTP {status}'
    return ods_data, None


def method_c_download(file_id, rev_num):
    """Method C: Direct Sheets export URL with revision number."""
    url = (f'https://docs.google.com/spreadsheets/export'
           f'?id={file_id}&revision={rev_num}&exportFormat=ods')
    status, ods_data = authenticated_get(url)
    if status != 200 or not ods_data or len(ods_data) < 500:
        return None, f'HTTP {status}'
    return ods_data, None


def ods_content_hash(ods_bytes):
    """Extract content.xml from ODS (ZIP) and SHA256 it."""
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            return hashlib.sha256(zf.read('content.xml')).hexdigest()
    except Exception:
        return None


def sha256(data):
    return hashlib.sha256(data).hexdigest()


print('Helpers ready')
print(f'CI data hashes: {len(CI_DATA_HASHES)} versions')
print(f'Content.xml refs: {sum(len(v) for v in CONTENT_XML_REFS.values())} files')

## Step 2: Direct Comparison — Method B vs Method C

Download the same 4 known-different revisions via both methods.
Compare:
- Raw ODS hash (expected to differ — non-deterministic metadata)
- content.xml hash (should match if both return same revision data)

In [ ]:
# Test 4 revisions where we KNOW content differs
TEST_CASES = [
    ('ubl25_library',   '1843', 'V1 state (initial)'),
    ('ubl25_library',   '2005', 'V7-V10 state (final)'),
    ('ubl25_documents', '1793', 'V1 state (initial)'),
    ('ubl25_documents', '2204', 'V9-V10 state (final)'),
]

print('Method B vs Method C: content.xml comparison')
print('='*75)

results = []
for sheet_key, rev_id, desc in TEST_CASES:
    file_id = SHEETS[sheet_key]
    print(f'\n--- {sheet_key} rev-{rev_id}: {desc} ---')

    # Method B
    print('  Method B (v2 exportLinks):', end=' ')
    b_data, b_err = method_b_download(file_id, rev_id)
    if b_data:
        b_full = sha256(b_data)
        b_content = ods_content_hash(b_data)
        print(f'{len(b_data):,} bytes, content={b_content[:24]}...')
    else:
        b_full = b_content = None
        print(f'FAILED: {b_err}')

    time.sleep(1)

    # Method C
    print('  Method C (direct URL):   ', end=' ')
    c_data, c_err = method_c_download(file_id, rev_id)
    if c_data:
        c_full = sha256(c_data)
        c_content = ods_content_hash(c_data)
        print(f'{len(c_data):,} bytes, content={c_content[:24]}...')
    else:
        c_full = c_content = None
        print(f'FAILED: {c_err}')

    # Compare
    if b_content and c_content:
        if b_content == c_content:
            print(f'  \u2705 content.xml MATCH (raw ODS: {"same" if b_full == c_full else "differ"})')
        else:
            print(f'  \u274c content.xml MISMATCH!')
            print(f'      B: {b_content[:32]}...')
            print(f'      C: {c_content[:32]}...')
    elif b_content:
        print(f'  \u26a0 Method C failed, cannot compare')
    elif c_content:
        print(f'  \u26a0 Method B failed, cannot compare')

    # Check against known reference
    ref = CONTENT_XML_REFS.get(sheet_key, {}).get(rev_id)
    if ref and b_content:
        match_b = b_content[:24] == ref
        print(f'  Ref check (B vs local): {"MATCH" if match_b else "DIFFER"}')
    if ref and c_content:
        match_c = c_content[:24] == ref
        print(f'  Ref check (C vs local): {"MATCH" if match_c else "DIFFER"}')

    results.append({
        'sheet': sheet_key, 'rev': rev_id, 'desc': desc,
        'b_content': b_content, 'c_content': c_content,
        'b_full': b_full, 'c_full': c_full,
        'match': b_content == c_content if b_content and c_content else None,
    })
    time.sleep(1)

# Summary
matched = sum(1 for r in results if r['match'] is True)
mismatched = sum(1 for r in results if r['match'] is False)
failed = sum(1 for r in results if r['match'] is None)
print(f'\n{"="*75}')
print(f'Summary: {matched} match, {mismatched} mismatch, {failed} failed')
if mismatched == 0 and failed == 0:
    print('\u2705 Methods B and C return identical content.xml for all tested revisions')
elif mismatched > 0:
    print('\u274c Methods B and C return DIFFERENT content! Investigate further.')

## Step 3: Method C Edge Cases

Test whether Method C returns genuinely historical content or falls back.

In [ ]:
# Edge case tests for Method C
file_id = SHEETS['ubl25_library']

print('Edge Case Tests for Method C (direct Sheets URL)')
print('='*75)

# Test 1: Download revision 1 (very first) - should be old content
print('\n--- Test 1: Revision 1 (first ever) ---')
rev1_data, rev1_err = method_c_download(file_id, '1')
if rev1_data:
    rev1_hash = ods_content_hash(rev1_data)
    latest_hash = ods_content_hash(method_c_download(file_id, '2005')[0])
    if rev1_hash != latest_hash:
        print(f'  \u2705 Rev 1 content DIFFERS from latest (rev 2005)')
        print(f'     Rev 1:    {rev1_hash[:32]}...')
        print(f'     Rev 2005: {latest_hash[:32]}...')
        print(f'     Rev 1 size: {len(rev1_data):,} bytes')
    else:
        print(f'  \u274c Rev 1 has SAME content as latest! Method C may be broken.')
else:
    print(f'  Rev 1 failed: {rev1_err}')

time.sleep(2)

# Test 2: Non-existent revision number
print('\n--- Test 2: Non-existent revision (99999) ---')
bad_data, bad_err = method_c_download(file_id, '99999')
if bad_data:
    bad_hash = ods_content_hash(bad_data)
    if bad_hash == latest_hash:
        print(f'  \u26a0 Non-existent rev returns LATEST content (silent fallback!)')
    else:
        print(f'  \u26a0 Non-existent rev returns SOME content: {bad_hash[:24]}...')
else:
    print(f'  \u2705 Non-existent rev correctly returns error: {bad_err}')

time.sleep(2)

# Test 3: Download same revision twice - check non-determinism
print('\n--- Test 3: Download rev 1843 twice (non-determinism test) ---')
d1, _ = method_c_download(file_id, '1843')
time.sleep(1)
d2, _ = method_c_download(file_id, '1843')
if d1 and d2:
    full_match = sha256(d1) == sha256(d2)
    content_match = ods_content_hash(d1) == ods_content_hash(d2)
    print(f'  Raw ODS: {"identical" if full_match else "different"}')
    print(f'  content.xml: {"identical" if content_match else "different"}')
    if content_match and not full_match:
        print(f'  \u2705 Expected: same content, different metadata')
    elif full_match:
        print(f'  Interesting: completely identical binary')
    else:
        print(f'  \u274c PROBLEM: content.xml differs for same revision!')

## Step 4: Find Correct Revisions for V2, V3, V6

Scan revision ranges around the known "major" revisions to find the
exact revision that matches each CI artifact's GC data hash.

Strategy: download ODS via Method C for each revision, convert to GC,
strip Identification, hash, compare against CI hashes.

In [ ]:
# Fetch tools from GitHub for ODS -> GC conversion
import subprocess, os, shutil, tempfile
from pathlib import Path

TOOLS_DIR = Path('/content/tools')
TOOLS_DIR.mkdir(exist_ok=True)
(TOOLS_DIR / 'support').mkdir(exist_ok=True)

BRANCH = 'claude/investigate-oasis-sheets-aYOlG'
RAW = f'https://raw.githubusercontent.com/kduvekot/ubl-gc/{BRANCH}'
CRANE = 'history/tools/Crane-ods2obdgc'

TOOL_URLS = {
    'saxon9he.jar':               f'{RAW}/history/tools/saxon9he/saxon9he.jar',
    'Crane-ods2obdgc.xsl':        f'{RAW}/{CRANE}/Crane-ods2obdgc.xsl',
    'support/gcExportSubset.xsl':  f'{RAW}/{CRANE}/support/gcExportSubset.xsl',
    'support/odsCommon.xsl':       f'{RAW}/{CRANE}/support/odsCommon.xsl',
    'massageModelName.xml':        f'{RAW}/work-sheets/scripts/massageModelName.xml',
    'gc2endorsed.xsl':            f'{RAW}/work-sheets/scripts/gc2endorsed.xsl',
}

for name, url in TOOL_URLS.items():
    dest = TOOLS_DIR / name
    if dest.exists() and dest.stat().st_size > 100:
        print(f'  [skip] {name}')
        continue
    print(f'  Downloading {name}...', end=' ')
    r = subprocess.run(['wget', '-q', '-O', str(dest), url],
                       capture_output=True, timeout=60)
    print(f'{dest.stat().st_size:,} bytes' if dest.exists() else 'FAILED')

SAXON_JAR   = str(TOOLS_DIR / 'saxon9he.jar')
CRANE_XSL   = str(TOOLS_DIR / 'Crane-ods2obdgc.xsl')
MASSAGE_XML = str(TOOLS_DIR / 'massageModelName.xml')
GC2ENDORSED = str(TOOLS_DIR / 'gc2endorsed.xsl')
SHEET_REGEX = r'^([Ll]($|[^o].*|o($|[^g].*|g($|[^s].*))))|^[^Ll].*'

!java -version 2>&1 | head -1
print('\nTools ready!')

In [ ]:
def convert_to_gc_and_hash(lib_ods_bytes, doc_ods_bytes):
    """Convert ODS pair -> GC, strip Identification, return data hash."""
    tmpdir = tempfile.mkdtemp()
    try:
        Path(f'{tmpdir}/UBL-Library-Google.ods').write_bytes(lib_ods_bytes)
        Path(f'{tmpdir}/UBL-Documents-Google.ods').write_bytes(doc_ods_bytes)
        shutil.copy2(MASSAGE_XML, f'{tmpdir}/massageModelName.xml')

        # Minimal ident file
        with open(f'{tmpdir}/ident.xml', 'w') as f:
            f.write('<?xml version="1.0"?>\n<Identification>'
                    '<ShortName>TEST</ShortName></Identification>')

        ods_list = f'{tmpdir}/UBL-Library-Google.ods,{tmpdir}/UBL-Documents-Google.ods'
        gc_out = f'{tmpdir}/entities.gc'

        result = subprocess.run([
            'java', '-jar', SAXON_JAR,
            f'-xsl:{CRANE_XSL}', f'-o:{gc_out}', '-it:ods-uri',
            f'ods-uri={ods_list}',
            f'identification-uri={tmpdir}/ident.xml',
            f'included-sheet-name-regex={SHEET_REGEX}',
            f'lengthen-model-name-uri={tmpdir}/massageModelName.xml',
        ], capture_output=True, text=True, timeout=180)

        if result.returncode != 0 or not os.path.exists(gc_out):
            return None, f'Saxon failed: {result.stderr[-200:]}'

        gc_text = Path(gc_out).read_text()
        stripped = re.sub(
            r'<Identification>.*?</Identification>\s*',
            '', gc_text, count=1, flags=re.DOTALL
        )
        return hashlib.sha256(stripped.encode()).hexdigest(), None
    except Exception as e:
        return None, str(e)
    finally:
        shutil.rmtree(tmpdir)


print('Conversion helper ready')

In [ ]:
# === SCAN FOR MISSING REVISIONS ===
#
# For each mismatched CI version, scan the revision range to find which
# revision number produces matching GC data.
#
# Strategy:
#   - For V2 (CI data hash c21c9fd6de75cfe5): library changed between
#     rev-1843 and rev-1868. Scan library revisions 1843-1870.
#     Documents didn't change (still rev-1793).
#
#   - For V3 (CI data hash 85d713818f2654c2): documents changed between
#     rev-1803 and rev-1983. But lib might also have changed.
#     Scan documents revisions 1800-1810 AND library 1860-1870.
#
#   - For V6 (CI data hash 1ebde1fcadd0f0f9): library changed between
#     rev-1999 and rev-2005. Scan library revisions 1999-2010.
#     Documents unchanged (still rev-2190).

# First, download the "partner" ODS files that DON'T change
print('Downloading partner ODS files...')

# Documents rev-1793 (partner for V2 library scan)
doc_1793, _ = method_c_download(SHEETS['ubl25_documents'], '1793')
print(f'  Documents rev-1793: {len(doc_1793):,} bytes' if doc_1793 else '  FAILED')
time.sleep(1)

# Documents rev-2190 (partner for V6 library scan)
doc_2190, _ = method_c_download(SHEETS['ubl25_documents'], '2190')
print(f'  Documents rev-2190: {len(doc_2190):,} bytes' if doc_2190 else '  FAILED')
time.sleep(1)

# Library rev-1868 (partner for V3 documents scan)
lib_1868, _ = method_c_download(SHEETS['ubl25_library'], '1868')
print(f'  Library rev-1868: {len(lib_1868):,} bytes' if lib_1868 else '  FAILED')

In [ ]:
# === SCAN 1: Find V2's library revision ===
# CI V2 data hash: c21c9fd6de75cfe5
# Library changed between rev-1843 (V1 match) and rev-1868 (V3 match)

TARGET_V2 = 'c21c9fd6de75cfe5'

print(f'=== Scanning for V2 (target: {TARGET_V2}...) ===')
print(f'Library revisions 1843-1870, paired with Documents rev-1793')
print()

v2_found = None
v2_scan_results = []

for rev in range(1843, 1871):
    print(f'  lib rev-{rev}:', end=' ', flush=True)

    lib_data, err = method_c_download(SHEETS['ubl25_library'], str(rev))
    if not lib_data:
        print(f'skip ({err})')
        time.sleep(0.3)
        continue

    content_hash = ods_content_hash(lib_data)
    print(f'{len(lib_data):,}b content={content_hash[:16]}...', end=' ')

    # Convert to GC with partner docs
    data_hash, conv_err = convert_to_gc_and_hash(lib_data, doc_1793)
    if data_hash:
        ci_match = CI_HASH_TO_VER.get(data_hash[:16])
        if ci_match:
            print(f'data={data_hash[:16]}... -> CI {ci_match} !!!')
            v2_scan_results.append({
                'rev': rev, 'data_hash': data_hash, 'match': ci_match,
                'content_hash': content_hash,
            })
            if TARGET_V2 in data_hash:
                v2_found = rev
        else:
            print(f'data={data_hash[:16]}... (no CI match)')
            v2_scan_results.append({
                'rev': rev, 'data_hash': data_hash, 'match': None,
                'content_hash': content_hash,
            })
    else:
        print(f'CONVERT FAILED: {conv_err}')

    time.sleep(0.5)

if v2_found:
    print(f'\n\u2705 V2 FOUND: library rev-{v2_found}')
else:
    print(f'\n\u274c V2 NOT FOUND in range 1843-1870')
    print('  Try expanding the range or checking documents changes too')

In [ ]:
# === SCAN 2: Find V3's documents revision ===
# CI V3 data hash: 85d713818f2654c2
# V3 used lib=1868 (same as our mapping), but doc was different from 1803
# Scan documents revisions around 1800-1810

TARGET_V3 = '85d713818f2654c2'

print(f'=== Scanning for V3 (target: {TARGET_V3}...) ===')
print(f'Documents revisions 1800-1990, paired with Library rev-1868')
print()

v3_found = None
v3_scan_results = []

for rev in range(1800, 1991):
    doc_data, err = method_c_download(SHEETS['ubl25_documents'], str(rev))
    if not doc_data:
        # Don't print for missing revisions (there are many gaps)
        time.sleep(0.3)
        continue

    content_hash = ods_content_hash(doc_data)
    print(f'  doc rev-{rev}: {len(doc_data):,}b content={content_hash[:16]}...', end=' ')

    data_hash, conv_err = convert_to_gc_and_hash(lib_1868, doc_data)
    if data_hash:
        ci_match = CI_HASH_TO_VER.get(data_hash[:16])
        if ci_match:
            print(f'data={data_hash[:16]}... -> CI {ci_match} !!!')
            v3_scan_results.append({
                'rev': rev, 'data_hash': data_hash, 'match': ci_match,
                'content_hash': content_hash,
            })
            if TARGET_V3 in data_hash:
                v3_found = rev
        else:
            print(f'data={data_hash[:16]}... (no CI match)')
            v3_scan_results.append({
                'rev': rev, 'data_hash': data_hash, 'match': None,
                'content_hash': content_hash,
            })
    else:
        print(f'CONVERT FAILED: {conv_err}')

    time.sleep(0.5)

if v3_found:
    print(f'\n\u2705 V3 FOUND: documents rev-{v3_found}')
else:
    print(f'\n\u274c V3 NOT FOUND in scanned range')
    print('  Unique data hashes found:', len(set(r['data_hash'] for r in v3_scan_results if r['data_hash'])))

In [ ]:
# === SCAN 3: Find V6's library revision ===
# CI V6 data hash: 1ebde1fcadd0f0f9
# Library changed between rev-1999 (V5 match) and rev-2005 (V7 match)

TARGET_V6 = '1ebde1fcadd0f0f9'

print(f'=== Scanning for V6 (target: {TARGET_V6}...) ===')
print(f'Library revisions 1999-2006, paired with Documents rev-2190')
print()

v6_found = None
v6_scan_results = []

for rev in range(1999, 2007):
    print(f'  lib rev-{rev}:', end=' ', flush=True)

    lib_data, err = method_c_download(SHEETS['ubl25_library'], str(rev))
    if not lib_data:
        print(f'skip ({err})')
        time.sleep(0.3)
        continue

    content_hash = ods_content_hash(lib_data)
    print(f'{len(lib_data):,}b content={content_hash[:16]}...', end=' ')

    data_hash, conv_err = convert_to_gc_and_hash(lib_data, doc_2190)
    if data_hash:
        ci_match = CI_HASH_TO_VER.get(data_hash[:16])
        if ci_match:
            print(f'data={data_hash[:16]}... -> CI {ci_match} !!!')
            v6_scan_results.append({
                'rev': rev, 'data_hash': data_hash, 'match': ci_match,
                'content_hash': content_hash,
            })
            if TARGET_V6 in data_hash:
                v6_found = rev
        else:
            print(f'data={data_hash[:16]}... (no CI match)')
            v6_scan_results.append({
                'rev': rev, 'data_hash': data_hash, 'match': None,
                'content_hash': content_hash,
            })
    else:
        print(f'CONVERT FAILED: {conv_err}')

    time.sleep(0.5)

if v6_found:
    print(f'\n\u2705 V6 FOUND: library rev-{v6_found}')
else:
    print(f'\n\u274c V6 NOT FOUND in range 1999-2006')
    print('  Try checking documents changes too')

## Step 5: Download Colab Manifests

The previous Colab run (`download-all-revisions.ipynb`) saved manifests
to Google Drive. These contain content.xml hashes and gc_data_hash for
ALL revisions processed. Let's retrieve them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
if DRIVE_DIR.exists():
    print(f'Found Drive directory: {DRIVE_DIR}')
    for f in sorted(DRIVE_DIR.iterdir()):
        if f.is_file():
            print(f'  {f.name}: {f.stat().st_size:,} bytes')
        elif f.is_dir():
            count = sum(1 for _ in f.iterdir())
            print(f'  {f.name}/ ({count} items)')
else:
    print('Drive directory not found (different Google account?)')
    print(f'Expected: {DRIVE_DIR}')

In [ ]:
# Read and analyze manifests
from collections import Counter

for sheet_key in ['ubl25_library', 'ubl25_documents']:
    mp = DRIVE_DIR / f'manifest-{sheet_key}.json'
    if not mp.exists():
        print(f'{sheet_key}: manifest not found')
        continue

    m = json.loads(mp.read_text())
    revs = m.get('revisions', [])

    print(f'\n{"="*60}')
    print(f'{sheet_key}: {len(revs)} revisions in manifest')
    print(f'{"="*60}')
    print(f'  Unique states: {m.get("unique_states", "?")}')
    print(f'  Converted to GC: {m.get("converted", "?")}')
    print(f'  Conversion errors: {m.get("conversion_errors", "?")}')

    # Find revisions with GC data hashes that match our CI targets
    print(f'\n  Checking GC data hashes against CI V1-V10...')
    for rev_entry in revs:
        gc_hash = rev_entry.get('gc_data_hash', '')
        if gc_hash:
            ci_match = CI_HASH_TO_VER.get(gc_hash[:16])
            if ci_match:
                print(f'    rev-{rev_entry["rev"]}: data={gc_hash[:16]}... '
                      f'-> matches CI {ci_match}')

    # Show hash stats
    hash_counts = Counter(r.get('content_hash') for r in revs if r.get('content_hash'))
    print(f'\n  Top 5 most common content states:')
    for h, count in hash_counts.most_common(5):
        first_rev = min(r['rev'] for r in revs if r.get('content_hash') == h)
        last_rev = max(r['rev'] for r in revs if r.get('content_hash') == h)
        print(f'    {h[:16]}... x{count} (rev {first_rev}-{last_rev})')

## Step 6: Summary & Conclusions

In [ ]:
print('='*75)
print('INVESTIGATION SUMMARY')
print('='*75)
print()

print('1. Method B vs C comparison:')
for r in results:
    status = 'MATCH' if r['match'] else ('MISMATCH' if r['match'] is False else 'FAILED')
    print(f'   {r["sheet"]} rev-{r["rev"]}: {status}')

print()
print('2. Missing revision search:')
if v2_found:
    print(f'   V2: library rev-{v2_found} (was mapped to 1843, should be {v2_found})')
else:
    print(f'   V2: NOT FOUND in scan range')
if v3_found:
    print(f'   V3: documents rev-{v3_found} (was mapped to 1803, should be {v3_found})')
else:
    print(f'   V3: NOT FOUND in scan range')
if v6_found:
    print(f'   V6: library rev-{v6_found} (was mapped to 1999, should be {v6_found})')
else:
    print(f'   V6: NOT FOUND in scan range')

print()
print('3. Updated revision mapping (copy to manifest):')
updated = dict(REV_MAP)
if v2_found:
    updated['V2'] = {**REV_MAP['V2'], 'lib': str(v2_found)}
if v3_found:
    updated['V3'] = {**REV_MAP['V3'], 'doc': str(v3_found)}
if v6_found:
    updated['V6'] = {**REV_MAP['V6'], 'lib': str(v6_found)}

for ver in ['V1','V2','V3','V4','V5','V6','V7','V8','V9','V10']:
    rev = updated[ver]
    changed = '***' if ver in ['V2','V3','V6'] else '   '
    print(f'   {changed} {ver}: lib={rev["lib"]}, doc={rev["doc"]}, stage={rev["stage"]}')